# Parameters

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd

# ===== CONFIGURAÇÃO DE CAMINHOS =====
current_notebook = Path.cwd()  
project_root = current_notebook.parent.parent 

# Adiciona o diretório raiz ao sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Adiciona o diretório Modules ao sys.path
modules_dir = project_root / "Modules"
if str(modules_dir) not in sys.path:
    sys.path.insert(0, str(modules_dir))

# ===== IMPORTS DOS MÓDULOS =====
import Modules.ClusterGmmModule as cluster
import Modules.FutureAnalysisModule as fa
from Modules.config import CONFIG

# ===== CONFIGURAÇÕES DO PROJETO =====
DATAPATH = CONFIG["datapath"]
COVID_TRAIN_DATA_FILE = CONFIG["covid_train_data_file"]
COVID_TEST_DATA_FILE = CONFIG["covid_test_data_file"]
FUTURE_DATA_FILE = CONFIG["future_data_file"]

CONTROL_GROUP_TRAIN = CONFIG["control_group_train"]
CONTROL_GROUP_TEST = CONFIG["control_group_test"]
CONTROL_GROUP_READMISSION = CONFIG["control_group_readmission"]

FIGSIZE_CLUSTER_HEATMAP = CONFIG["figsize_cluster_heatmap"]
FIGSIZE_FUTURE_HEATMAP = CONFIG["figsize_future_heatmap"]
IMAGES_SAVE_PATH = CONFIG["image_save_path"]
TRIALS_OPTUNA = 80

# Import data

In [2]:
# ===== CARREGAMENTO DOS DADOS =====
data_folder = current_notebook / DATAPATH

covid_train = pd.read_csv(data_folder / COVID_TRAIN_DATA_FILE)
covid_test = pd.read_csv(data_folder / COVID_TEST_DATA_FILE)
future_data = pd.read_csv(data_folder / FUTURE_DATA_FILE)

control_train = pd.read_csv(data_folder / CONTROL_GROUP_TRAIN)
control_test = pd.read_csv(data_folder / CONTROL_GROUP_TEST)
control = pd.concat([control_train, control_test], axis=0)
control_readmission = pd.read_csv(data_folder / CONTROL_GROUP_READMISSION)

# Feature engineering: morte após a internação
covid_train['died_after'] = ((covid_train['died'] == 1) & (covid_train['died_in_stay'] == 0)).astype(int)
covid_test['died_after'] = ((covid_test['died'] == 1) & (covid_test['died_in_stay'] == 0)).astype(int)
future_data['died_after'] = ((future_data['died'] == 1) & (future_data['died_in_stay'] == 0)).astype(int)

In [3]:
data_covid = pd.concat([covid_train, covid_test], axis=0)
data_covid = data_covid.sample(frac=1, random_state=42).reset_index(drop=True)

# Mice Data

In [4]:
categorical_features = [
            "myocardial_infarct",
            "congestive_heart_failure",
            "peripheral_vascular_disease",
            "cerebrovascular_disease",
            "dementia",
            "chronic_pulmonary_disease",
            "rheumatic_disease",
            "peptic_ulcer_disease",
            "mild_liver_disease",
            "diabetes_without_cc",
            "diabetes_with_cc",
            "paraplegia",
            "renal_disease",
            "malignant_cancer",
            "severe_liver_disease",
            "metastatic_solid_tumor",
            "aids",
            "gender_M",
            "died_in_stay",
            "died_after",
            "died",
            "COVID"
        ]

In [5]:
featuresNotConsidered = ['died', 'died_in_stay', 'died_after', 'COVID', 'subject_id', 'hadm_id']

In [6]:
helper = cluster.GmmClusterHelper(data=data_covid, featuresNotConsidered=featuresNotConsidered)

## Find best hyperparameters for GMM

In [7]:
param = {
    "n_components": {"min": 2, "max": 15},
    "covariance_type": ["full", "tied", "diag", "spherical"]
}

### DBCV

In [8]:
# os.environ["PYTHONWARNINGS"] = "ignore"
# pca_dbcv_df, pca_dbcv_param, pca_dbcv_best = helper.optunaGridSearch(
#     parameters=param, 
#     n_trials=TRIALS_OPTUNA, 
#     saveStorage=True, 
#     metric="dbcv",
#     suffix="pca",
#     restrictMinsizeCluster=2,
#     restrictMaxsizeCluster=80,
#     dimensionality_reduction = {'method': 'PCA', 'dimensions': 30}
# )

In [9]:
pca_dbcv_param = {'n_components': 11, 'covariance_type': 'full'}

In [10]:
# os.environ["PYTHONWARNINGS"] = "ignore"
# ae_dbcv_df, ae_dbcv_param, ae_dbcv_best = helper.optunaGridSearch(
#     parameters=param, 
#     n_trials=TRIALS_OPTUNA, 
#     saveStorage=True, 
#     metric="dbcv",
#     suffix="ae",
#     restrictMinsizeCluster=2,
#     restrictMaxsizeCluster=80,
#     dimensionality_reduction = {'method': 'AE', 'dimensions': 10}
# )

In [11]:
ae_dbcv_param = {'n_components': 12, 'covariance_type': 'diag'}

### DISCO

In [12]:
# os.environ["PYTHONWARNINGS"] = "ignore"
# pca_disco_df, pca_disco_param, pca_disco_best = helper.optunaGridSearch(
#     parameters=param, 
#     n_trials=TRIALS_OPTUNA, 
#     saveStorage=True, 
#     metric="disco",
#     suffix="pca",
#     restrictMinsizeCluster=2,
#     restrictMaxsizeCluster=80,
#     dimensionality_reduction = {'method': 'PCA', 'dimensions': 30}
# )

In [13]:
pca_disco_param = {'n_components': 2, 'covariance_type': 'diag'}

In [14]:
# os.environ["PYTHONWARNINGS"] = "ignore"
# ae_disco_df, ae_disco_param, ae_disco_best = helper.optunaGridSearch(
#     parameters=param, 
#     n_trials=TRIALS_OPTUNA, 
#     saveStorage=True, 
#     metric="disco",
#     suffix="ae",
#     restrictMinsizeCluster=2,
#     restrictMaxsizeCluster=80,
#     dimensionality_reduction = {'method': 'AE', 'dimensions': 10}
# )

In [15]:
ae_disco_param = {'n_components': 2, 'covariance_type': 'diag'}

### DSI

In [16]:
# os.environ["PYTHONWARNINGS"] = "ignore"
# pca_dsi_df, pca_dsi_param, pca_dsi_best = helper.optunaGridSearch(
#     parameters=param, 
#     n_trials=TRIALS_OPTUNA, 
#     saveStorage=True, 
#     metric="dsi",
#     suffix="pca",
#     restrictMinsizeCluster=2,
#     restrictMaxsizeCluster=80,
#     dimensionality_reduction = {'method': 'PCA', 'dimensions': 30}
# )

In [17]:
pca_dsi_param = {'n_components': 2, 'covariance_type': 'spherical'}

In [18]:
# os.environ["PYTHONWARNINGS"] = "ignore"
# ae_dsi_df, ae_dsi_param, ae_dsi_best = helper.optunaGridSearch(
#     parameters=param, 
#     n_trials=TRIALS_OPTUNA, 
#     saveStorage=True, 
#     metric="dsi",
#     suffix="ae",
#     restrictMinsizeCluster=2,
#     restrictMaxsizeCluster=80,
#     dimensionality_reduction = {'method': 'AE', 'dimensions': 10}
# )

In [19]:
ae_dsi_param = {'n_components': 2, 'covariance_type': 'diag'}

### Silhouette

In [20]:
# os.environ['PYTHONWARNINGS'] = 'ignore'
# pca_silhouette_df, pca_silhouette_param, pca_silhouette_best = helper.optunaGridSearch(
#     parameters=param, 
#     n_trials=TRIALS_OPTUNA, 
#     saveStorage=True, 
#     metric="silhouette",
#     suffix="pca",
#     restrictMinsizeCluster=2,
#     restrictMaxsizeCluster=80,
#     dimensionality_reduction = {'method': 'PCA', 'dimensions': 30}
# )

In [21]:
pca_silhouette_param = {'n_components': 2, 'covariance_type': 'spherical'}

In [22]:
# os.environ['PYTHONWARNINGS'] = 'ignore'
# ae_silhouette_df, ae_silhouette_param, ae_silhouette_best = helper.optunaGridSearch(
#     parameters=param, 
#     n_trials=TRIALS_OPTUNA, 
#     saveStorage=True, 
#     metric="silhouette",
#     suffix="ae",
#     restrictMinsizeCluster=2,
#     restrictMaxsizeCluster=80,
#     dimensionality_reduction = {'method': 'AE', 'dimensions': 10}
# )

In [23]:
ae_silhouette_param = {'n_components': 2, 'covariance_type': 'diag'}

### Metrics

In [24]:
helper.clustering(
    n_components=pca_dbcv_param["n_components"],
    covariance_type=pca_dbcv_param["covariance_type"],
    dimensionality_reduction = {'method': 'PCA', 'dimensions': 30}
)
helper.getMetrics()

/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)


{'silhouette': -0.249,
 'dbcv': -0.7491438752928511,
 'dsi': np.float64(0.185),
 'disco': np.float64(-0.14938400796869353)}

In [34]:
helper.clustering(
    n_components=pca_disco_param["n_components"],
    covariance_type=pca_disco_param["covariance_type"],
    dimensionality_reduction={"method": "PCA", "dimensions": 30},
)
helper.getMetrics()

/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)


{'silhouette': 0.352,
 'dbcv': -0.9912368830216302,
 'dsi': np.float64(0.195),
 'disco': np.float64(0.40004798331902336)}

In [26]:
helper.clustering(
    n_components=pca_dsi_param["n_components"],
    covariance_type=pca_dsi_param["covariance_type"],
    dimensionality_reduction={"method": "PCA", "dimensions": 30},
)
helper.getMetrics()

/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)


{'silhouette': 0.356,
 'dbcv': -0.9931120058343926,
 'dsi': np.float64(0.205),
 'disco': np.float64(0.3981893618981151)}

In [27]:
helper.clustering(
    n_components=pca_silhouette_param["n_components"],
    covariance_type=pca_silhouette_param["covariance_type"],
    dimensionality_reduction={"method": "PCA", "dimensions": 30},
)
helper.getMetrics()

/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)


{'silhouette': 0.356,
 'dbcv': -0.9931120058343926,
 'dsi': np.float64(0.205),
 'disco': np.float64(0.3981893618981151)}

In [28]:
helper.clustering(
    n_components=ae_dbcv_param["n_components"],
    covariance_type=ae_dbcv_param["covariance_type"],
    dimensionality_reduction={"method": "AE", "dimensions": 10},
)
helper.getMetrics()

Autoencoder is not fitted yet, will be pretrained.
Start training with clustering loss.
Epoch: 100%|██████████| 100/100 [00:25<00:00,  3.93it/s]


/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)


{'silhouette': -0.197,
 'dbcv': -0.8121189580077216,
 'dsi': np.float64(0.1),
 'disco': np.float64(-0.1839974132146546)}

In [29]:
helper.clustering(
    n_components=ae_disco_param["n_components"],
    covariance_type=ae_disco_param["covariance_type"],
    dimensionality_reduction={"method": "AE", "dimensions": 10},
)
helper.getMetrics()

Autoencoder is not fitted yet, will be pretrained.
Start training with clustering loss.
Epoch: 100%|██████████| 100/100 [00:28<00:00,  3.45it/s]


/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)


{'silhouette': 0.295,
 'dbcv': -0.9950213863645341,
 'dsi': np.float64(0.189),
 'disco': np.float64(0.335481287014478)}

In [30]:
helper.clustering(
    n_components=ae_dsi_param["n_components"],
    covariance_type=ae_dsi_param["covariance_type"],
    dimensionality_reduction={"method": "AE", "dimensions": 10},
)
helper.getMetrics()

Autoencoder is not fitted yet, will be pretrained.
Start training with clustering loss.
Epoch: 100%|██████████| 100/100 [00:29<00:00,  3.40it/s]


/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)


{'silhouette': 0.295,
 'dbcv': -0.9950213863645341,
 'dsi': np.float64(0.189),
 'disco': np.float64(0.335481287014478)}

In [31]:
helper.clustering(
    n_components=ae_silhouette_param["n_components"],
    covariance_type=ae_silhouette_param["covariance_type"],
    dimensionality_reduction={"method": "AE", "dimensions": 10},
)
helper.getMetrics()

Autoencoder is not fitted yet, will be pretrained.
Start training with clustering loss.
Epoch: 100%|██████████| 100/100 [00:29<00:00,  3.36it/s]


/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)
/Users/gabrielleite/Backup/Mestrado/EnvMestrado/lib/python3.11/site-packages/dbcv/core.py:91: RuntimeWarning: divide by zero encountered in power
  np.power(core_dists, -1.0 / d, out=core_dists)


{'silhouette': 0.295,
 'dbcv': -0.9950213863645341,
 'dsi': np.float64(0.189),
 'disco': np.float64(0.335481287014478)}

In [32]:
adsdasd

NameError: name 'adsdasd' is not defined

## Best

In [ ]:
best_pamram, best_dr = (pca_silhouette_param, {'method': 'AE', 'dimensions': 10})

In [ ]:
helper.clustering(
    n_components=best_pamram["n_components"],
    covariance_type=best_pamram["covariance_type"],
    dimensionality_reduction=best_dr,
)
helper.heatmapClustersCategorical(
    figsize=FIGSIZE_CLUSTER_HEATMAP,
    savepath=IMAGES_SAVE_PATH + "gmm-dr-categorical",
)

In [ ]:
helper.showClusterCompareNumerical(
    scaled="standard",
    topFeatures=-1,
    max_features=-1,
    minimumClusterSize=5,
    figsize=(10, 40),
    savepath=IMAGES_SAVE_PATH + "gmm-dr-numerical", verbose=1)

In [ ]:
selectedClusters = [0, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12]

In [ ]:
helper.setClusteredAutoencoder()

In [ ]:
helper.showClusteredAutoencoder(selectedClusters=selectedClusters, savepath=IMAGES_SAVE_PATH + "gmm-autoencoder-dr")

### Future Data

In [ ]:
future_helper = fa.FutureAnalysisHelper(helper.clusteredData, future_data)
future_helper.insertClustersInFutureData(onlyFirstAdmission=True)
delta = future_helper.getDeltaClusters(percentage=True, metric="interno")
future_helper.showDeltaHeatmap(
    figsize=FIGSIZE_FUTURE_HEATMAP,
    savepath=IMAGES_SAVE_PATH + "gmm-dr-interno", selectedClusters=selectedClusters,
    metric="interno"
)
delta = future_helper.getDeltaClusters(percentage=True, metric="externo")
future_helper.showDeltaHeatmap(
    figsize=FIGSIZE_FUTURE_HEATMAP,
    savepath=IMAGES_SAVE_PATH + "gmm-dr-externo", selectedClusters=selectedClusters,
    metric="externo"
)

In [ ]:
future_helper.getMeanReadmission()

In [ ]:
future_helper.getMeanDaysGap()

In [ ]:
future_helper.getMortalityRates(onlyFirstAdmission=True)

# Add Log

In [ ]:
metrics = helper.getMetrics()

log_file = "../log.csv"
current_dir = os.getcwd()
log_file_path = os.path.join(current_dir, log_file)

# Add line to save log
if os.path.exists(log_file_path):
    with open(log_file_path, 'a') as f:
        f.write(f"GMM, Autoencoder, Comprehensive, {metrics['disco']}, {metrics['dbcv']}, {metrics['dsi']}, {metrics['silhouette']}\n")